# 第10章: 事前学習済み言語モデル（GPT型）

本章では、GPT型（Transformerのデコーダ型）の事前学習済みモデルを利用して、言語生成、評判分析器（ポジネガ分類器）の構築、ファインチューニング、強化学習などに取り組む。

## 90. 次単語予測

“The movie was full of"に続くトークン（トークン列ではなく一つのトークンであることに注意せよ）として適切なもの上位10個と、その確率（尤度）を求めよ。ただし、言語モデルへのプロンプトがどのようなトークン列に変換されたか、確認せよ。

In [1]:
!pip install torch transformers datasets accelerate trl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 761.1/761.1 kB 25.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 23.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 16.9 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0


In [2]:
import torch
from transformers import GPT2LMHeadModel, GPT2Tokenizer
from transformers import AutoModelForCausalLM, AutoTokenizer, AutoModelForSequenceClassification
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling
from datasets import load_dataset
from tqdm import tqdm
import numpy as np
from trl import DPOTrainer, DPOConfig

In [3]:
# 1. モデルとトークナイザーのロード
model_id = "gpt2"
print(f"{model_id} モデルをロード中...\n")
tokenizer = GPT2Tokenizer.from_pretrained(model_id)
model = GPT2LMHeadModel.from_pretrained(model_id)

gpt2 モデルをロード中...



/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [ ]:
# 評価モードに設定（推論時はDropoutなどを無効化するため必須）
model.eval()

# 2. 入力テキストのトークン化と確認
text = "The movie was full of"
inputs = tokenizer(text, return_tensors="pt")
input_ids = inputs["input_ids"][0]

print("=== トークン化の確認 ===")
print(f"入力テキスト: '{text}'\n")

# input_idsを人間が読めるトークン文字列に変換して確認
tokens = tokenizer.convert_ids_to_tokens(input_ids)
for token_id, token_str in zip(input_ids, tokens):
    # GPT-2のBPEトークナイザーでは、単語の前の空白は 'Ġ' として表現される
    readable_token = token_str.replace('Ġ', ' (space)')
    print(f"Token ID: {token_id.item():<5} | Token: '{readable_token}'")

# 3. 次のトークンの予測
# 勾配計算を無効化してメモリ消費を抑え、推論を高速化
with torch.no_grad():
    outputs = model(**inputs)

# ロジット（未正規化のスコア）を取得
# 形状: (batch_size, sequence_length, vocab_size)
# ここでは最後のトークンの次に来る予測結果（[-1]）を取得
next_token_logits = outputs.logits[0, -1, :]

# ソフトマックス関数を適用して確率（尤度）に変換
probs = torch.nn.functional.softmax(next_token_logits, dim=-1)

# 4. 上位10個のトークンとその確率を取得
top_k = 10
top_k_probs, top_k_indices = torch.topk(probs, top_k)

print("\n=== 'The movie was full of' に続くトークン 上位10個 ===")
for i in range(top_k):
    token_id = top_k_indices[i].item()
    prob = top_k_probs[i].item()
    # IDを元のテキスト表現にデコード
    decoded_token = tokenizer.decode([token_id])

    print(f"{i+1:>2}. Token: '{decoded_token}' (ID: {token_id:<5}) | 確率: {prob:.4f} ({prob*100:.2f}%)")

=== トークン化の確認 ===
入力テキスト: 'The movie was full of'

Token ID: 464   | Token: 'The'
Token ID: 3807  | Token: ' (space)movie'
Token ID: 373   | Token: ' (space)was'
Token ID: 1336  | Token: ' (space)full'
Token ID: 286   | Token: ' (space)of'

=== 'The movie was full of' に続くトークン 上位10個 ===
 1. Token: ' jokes' (ID: 14532) | 確率: 0.0219 (2.19%)
 2. Token: ' great' (ID: 1049 ) | 確率: 0.0186 (1.86%)
 3. Token: ' laughs' (ID: 22051) | 確率: 0.0115 (1.15%)
 4. Token: ' bad' (ID: 2089 ) | 確率: 0.0109 (1.09%)
 5. Token: ' surprises' (ID: 24072) | 確率: 0.0107 (1.07%)
 6. Token: ' references' (ID: 10288) | 確率: 0.0105 (1.05%)
 7. Token: ' fun' (ID: 1257 ) | 確率: 0.0100 (1.00%)
 8. Token: ' humor' (ID: 14733) | 確率: 0.0074 (0.74%)
 9. Token: ' "' (ID: 366  ) | 確率: 0.0074 (0.74%)
10. Token: ' the' (ID: 262  ) | 確率: 0.0067 (0.67%)


## 91. 続きのテキストの予測

“The movie was full of"に続くテキストを複数予測せよ。このとき、デコーディングの方法や温度パラメータ（temperature）を変えながら、予測される複数のテキストの変化を観察せよ。

In [5]:
model.eval()

prompt = "The movie was full of"
inputs = tokenizer(prompt, return_tensors="pt")

# パディングトークンの設定
# 文の終わりを示すトークン（EOS）をパディング用のトークンとして代用
tokenizer.pad_token = tokenizer.eos_token

# 生成時の共通パラメータ
base_kwargs = {
    "input_ids": inputs["input_ids"],               # トークンのID
    "attention_mask": inputs["attention_mask"],     # トークンは1, パディングは0のフラグ
    "max_length": 50,                               # プロンプトを含めて最大50トークンまで生成する
    "pad_token_id": tokenizer.eos_token_id,         # PADとしてEOSを使う
    "no_repeat_ngram_size": 2,                      # 直前の2つの単語（2-gram）の連続が、生成文の中で二度と繰り返されないようにする
}

# 比較する
strategies = [
    {
        "name": "1. Greedy Search (貪欲探索)",
        "desc": "常に確率が最大のトークンを選択。ランダム性はなく、毎回同じ結果になります。",
        "kwargs": {"do_sample": False}
    },
    {
        "name": "2. Beam Search (ビーム探索: beams=5)",
        "desc": "複数の候補を保持しながら探索。Greedyより文脈として自然になりやすいです。",
        "kwargs": {"do_sample": False, "num_beams": 5}
    },
    {
        "name": "3. Sampling (Temperature = 0.5)",
        "desc": "低い温度。確率が高い単語がより強調され、保守的で安全な文章が生成されます。",
        "kwargs": {"do_sample": True, "temperature": 0.5, "top_k": 0}
    },
    {
        "name": "4. Sampling (Temperature = 1.0)",
        "desc": "デフォルトの温度。モデルが予測した元の確率分布のままサンプリングします。",
        "kwargs": {"do_sample": True, "temperature": 1.0, "top_k": 0}
    },
    {
        "name": "5. Sampling (Temperature = 1.5)",
        "desc": "高い温度。確率が低い単語も選ばれやすくなり、多様性が増しますが文脈が破綻しやすくなります。",
        "kwargs": {"do_sample": True, "temperature": 1.5, "top_k": 0}
    },
    {
        # 確率が高い順に単語を足していき、合計確率が90%（0.9）に達するまでの単語グループ（上位層）からサンプリング
        "name": "6. Top-p (Nucleus) Sampling (p=0.9)",
        "desc": "累積確率が p (0.9) を超えるまでの上位トークン群からサンプリング。品質と多様性のバランスが良いです。",
        "kwargs": {"do_sample": True, "top_p": 0.9, "top_k": 0}
    }
]

print(f"=== プロンプト ===\n'{prompt}'\n")

# 比較しやすいように乱数シードを固定
torch.manual_seed(42)

for strategy in strategies:
    print(f"--- {strategy['name']} ---")
    print(f"特徴: {strategy['desc']}")

    # 生成時の共通パラメータをコピーする（直接代入すると、元のbase_kwargsも変わってしまうため）
    kwargs = base_kwargs.copy()
    # 固有の設定を上書き・追加
    kwargs.update(strategy["kwargs"])

    # ランダム性がある手法、またはビームサーチの場合は3つの異なる出力を生成
    if (kwargs.get("do_sample") or kwargs.get("num_beams", 1) > 1):
        kwargs["num_return_sequences"] = 3
    else:
        kwargs["num_return_sequences"] = 1 # Greedyは常に1つ

    # 勾配計算を無効化して、テキスト生成を実行
    with torch.no_grad():
        outputs = model.generate(**kwargs)

    # 結果の出力
    for i, output in enumerate(outputs):
        decoded_text = tokenizer.decode(output, skip_special_tokens=True)
        print(f"  [{i+1}] {decoded_text}")

    print()

=== プロンプト ===
'The movie was full of'

--- 1. Greedy Search (貪欲探索) ---
特徴: 常に確率が最大のトークンを選択。ランダム性はなく、毎回同じ結果になります。
  [1] The movie was full of jokes and jokes about how the movie would be a "fantasy" and how it would have to be "real" to make it work.

The film was also full with jokes that were not true. For example

--- 2. Beam Search (ビーム探索: beams=5) ---
特徴: 複数の候補を保持しながら探索。Greedyより文脈として自然になりやすいです。
  [1] The movie was full of jokes and jokes, and it was funny, but it wasn't funny at all. It was like, "Oh, I'm not going to do this. I don't want to be a part of this."


  [2] The movie was full of jokes and jokes, and it was funny, but it wasn't funny at all. It was like, "Oh, I'm not going to do this. I don't want to be a part of this movie."

  [3] The movie was full of jokes and jokes, and it was funny, but it wasn't funny at all. It was like, "Oh, I'm not going to do this. I don't want to be a part of this movie." And

--- 3. Sampling (Temperature = 0.5) ---
特徴: 低い温度。確率が高い単語がより強調され、保

## 92. 予測されたテキストの確率を計算

“The movie was full of"に続くテキストを予測し、生成された各単語の尤度を表示せよ（生成されるテキストが長いと出力が読みにくくなるので、適当な長さで生成を打ち切るとよい）。

In [ ]:
model.eval()

# 2. プロンプトの準備
prompt = "The movie was full of"
inputs = tokenizer(prompt, return_tensors="pt")
input_length = inputs["input_ids"].shape[1]

# 3. テキストの生成とスコアの取得
max_new_tokens = 10  # 読みやすさのため10トークンで生成を打ち切る

with torch.no_grad():
    # return_dict_in_generate=True と output_scores=True を指定することで、
    # 生成されたトークンごとのロジット（未正規化スコア）を取得できます。
    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        output_scores=True,
        return_dict_in_generate=True,
        do_sample=False, # 確率が最も高いものを常に選ぶ（Greedy Search）
        pad_token_id=tokenizer.eos_token_id
    )

# 4. 結果の抽出
# 生成された全体のトークンID
generated_sequence = outputs.sequences[0]
# プロンプト部分を除外し、新しく生成されたトークンIDのみを抽出
new_tokens = generated_sequence[input_length:]

print(f"=== プロンプト ===\n'{prompt}'\n")

# 全体の生成テキストを表示
full_text = tokenizer.decode(generated_sequence, skip_special_tokens=True)
print(f"=== 生成された全体テキスト ===\n'{full_text}'\n")

print("=== 各トークンの生成確率（尤度） ===")
print(f"{'Step':<5} | {'Token':<15} | {'Probability':<10}")
print("-" * 40)

# 5. 各ステップの確率を計算して表示
# outputs.scores は各生成ステップごとのロジットのタプルです
for i, token_id in enumerate(new_tokens):
    # i番目のステップのロジットを取得 (形状: [batch_size, vocab_size])
    logits = outputs.scores[i][0]

    # ソフトマックス関数でロジットを確率分布（0.0〜1.0）に変換
    probs = torch.nn.functional.softmax(logits, dim=-1)

    # 実際に選ばれたトークンの確率を取得
    token_prob = probs[token_id].item()

    # トークンIDを文字列にデコード（空白などがわかるようにrepr()で表示）
    token_str = tokenizer.decode([token_id])
    token_str_display = repr(token_str)

    print(f"{i+1:<5} | {token_str_display:<15} | {token_prob*100:>6.2f}%")

=== プロンプト ===
'The movie was full of'

=== 生成された全体テキスト ===
'The movie was full of jokes and jokes about how the movie was a joke'

=== 各トークンの生成確率（尤度） ===
Step  | Token           | Probability
----------------------------------------
1     | ' jokes'        |   2.19%
2     | ' and'          |  28.92%
3     | ' jokes'        |   9.85%
4     | ' about'        |  20.56%
5     | ' how'          |   9.97%
6     | ' the'          |   8.46%
7     | ' movie'        |   3.64%
8     | ' was'          |  29.63%
9     | ' a'            |   6.77%
10    | ' joke'         |  17.35%


## 93. パープレキシティ

適当な文を準備して、事前学習済み言語モデルでパープレキシティを測定せよ。例えば、

+ The movie was full of surprises
+ The movies were full of surprises
+ The movie were full of surprises
+ The movies was full of surprises

の4文に対して、パープレキシティを測定して観察せよ（最後の2つの文は故意に文法的な間違いを入れた）。

In [ ]:
# 推論モードに設定
model.eval()

# 2. 評価する文のリスト（上2つが正解、下2つが文法エラー）
sentences = [
    "The movie was full of surprises",    # 単数 - 単数 (正)
    "The movies were full of surprises",  # 複数 - 複数 (正)
    "The movie were full of surprises",   # 単数 - 複数 (誤)
    "The movies was full of surprises"    # 複数 - 単数 (誤)
]

print("=== パープレキシティの測定結果 ===")
print(f"{'Perplexity':>10} | {'Sentence'}")
print("-" * 55)

# 3. 各文のパープレキシティを計算
for sentence in sentences:
    # テキストをトークン化し、PyTorchのテンソルに変換
    inputs = tokenizer(sentence, return_tensors="pt")
    input_ids = inputs["input_ids"]

    with torch.no_grad():
        # labelsにinput_idsを渡すことで、内部的に次のトークンを予測する
        # クロスエントロピー誤差（loss）が自動的に計算されます。
        outputs = model(input_ids, labels=input_ids)
        loss = outputs.loss

        # Lossの指数関数をとってパープレキシティを算出
        perplexity = torch.exp(loss)

    print(f"{perplexity.item():>10.2f} | {sentence}")

=== パープレキシティの測定結果 ===
Perplexity | Sentence
-------------------------------------------------------


[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


     99.35 | The movie was full of surprises
    126.48 | The movies were full of surprises
    278.88 | The movie were full of surprises
    274.66 | The movies was full of surprises


## 94. チャットテンプレート

"What do you call a sweet eaten after dinner?"という問いかけに対する応答を生成するため、チャットテンプレートを適用し、言語モデルに与えるべきプロンプトを作成せよ。また、そのプロンプトに対する応答を生成し、表示せよ。

In [ ]:
# 1. チャット対応モデルとトークナイザーのロード
# 軽量でチャットテンプレートをサポートしているTinyLlamaを使用
chat_model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
print(f"{chat_model_id} モデルをロード中...\n")

chat_tokenizer = AutoTokenizer.from_pretrained(chat_model_id)
chat_model = AutoModelForCausalLM.from_pretrained(chat_model_id)

TinyLlama/TinyLlama-1.1B-Chat-v1.0 モデルをロード中...



config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.29k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [ ]:
chat_model.eval()

# 2. 会話履歴（メッセージ）の定義
# "system" ロールでAIの振る舞いを指定し、"user" ロールでユーザーの入力を指定
messages = [
    {"role": "system", "content": "You are a helpful and polite AI assistant."},
    {"role": "user", "content": "What do you call a sweet eaten after dinner?"}
]

# 3. チャットテンプレートの適用
# messagesを特殊トークンで囲み、システム命令と質問を分ける
prompt_string = chat_tokenizer.apply_chat_template(
    messages,
    tokenize=False,               # tokenize=False を指定することで、トークンIDのリストではなく文字列として取得できる
    add_generation_prompt=True    # add_generation_prompt=True は、最後にAIの応答開始を示す特殊トークンを追加する
)

print("=== 言語モデルに与えられる実際のプロンプト ===")
print(prompt_string)
print("============================================\n")

# 4. モデルへの入力の準備（文字列をトークン化）
inputs = chat_tokenizer(prompt_string, return_tensors="pt")

# 5. 応答の生成
with torch.no_grad():
    outputs = chat_model.generate(
        **inputs,
        max_new_tokens=50,       # 生成する最大トークン数
        do_sample=True,          # サンプリングを有効化
        temperature=0.7,         # 温度パラメータ
        top_p=0.9,               # Top-pサンプリング
        pad_token_id=chat_tokenizer.eos_token_id
    )

# 6. 生成された結果からAIの応答部分のみを抽出してデコード
# outputsには、プロンプトと出力が含まれているため、プロンプトの長さを取得し、新しく生成された部分だけを切り出します
input_length = inputs["input_ids"].shape[1]
generated_tokens = outputs[0][input_length:]

# トークンIDを単語に変換する
# skip_special_tokens=Trueで、システム文字を消す
response_text = chat_tokenizer.decode(generated_tokens, skip_special_tokens=True)

print("=== AIの応答 ===")
print(response_text)

[transformers] Both `max_new_tokens` (=50) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


=== 言語モデルに与えられる実際のプロンプト ===
<|system|>
You are a helpful and polite AI assistant.</s>
<|user|>
What do you call a sweet eaten after dinner?</s>
<|assistant|>


=== AIの応答 ===
A sweet is called a dessert.


## 95. マルチターンのチャット

問題94で生成された応答に対して、追加で"Please give me the plural form of the word with its spelling in reverse order."と問いかけたときの応答を生成・表示せよ。また、その時に言語モデルに与えるプロンプトを確認せよ。

In [ ]:
# ==========================================
# 2ターン目：追加の問いかけ
# ==========================================
# 重要なステップ: 1ターン目のAIの応答を 'assistant' ロールとして履歴に追加する
# messages.append({"role": "assistant", "content": response_text_1})

# さらに、今回の新しい問いかけを 'user' ロールとして追加する
new_question = "Please give me the plural form of the word with its spelling in reverse order."
messages.append({"role": "user", "content": new_question})

# 2ターン目用のプロンプトを作成（これまでの全履歴が含まれる）
prompt_2 = chat_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

print("=== 言語モデルに与えられるプロンプト（2ターン目） ===")
print(prompt_2)
print("===================================================\n")

# 2ターン目の生成
inputs_2 = chat_tokenizer(prompt_2, return_tensors="pt")

with torch.no_grad():
    outputs_2 = chat_model.generate(
        **inputs_2,
        max_new_tokens=50,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
        pad_token_id=chat_tokenizer.eos_token_id
    )

input_length_2 = inputs_2["input_ids"].shape[1]
response_text_2 = chat_tokenizer.decode(outputs_2[0][input_length_2:], skip_special_tokens=True).strip()

print("=== 2ターン目のAI応答 ===")
print(response_text_2)

[transformers] Both `max_new_tokens` (=50) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


=== 言語モデルに与えられるプロンプト（2ターン目） ===
<|system|>
You are a helpful and polite AI assistant.</s>
<|user|>
What do you call a sweet eaten after dinner?</s>
<|user|>
Please give me the plural form of the word with its spelling in reverse order.</s>
<|assistant|>


=== 2ターン目のAI応答 ===
The plural form of "with its spelling in reverse order" is "with its spelling in reverse order".


## 96. プロンプトによる感情分析

事前学習済み言語モデルで感情分析を行いたい。テキストを含むプロンプトを事前学習済み言語モデルに与え、（ファインチューニングは行わずに）テキストのポジネガを予測するという戦略で、[SST-2](https://dl.fbaipublicfiles.com/glue/data/SST-2.zip)の開発データにおける正解率を測定せよ。

In [ ]:
model.eval()

# 2. SST-2データセットのロード (validation splitを使用)
print("SST-2 データセットをロード中...")
dataset = load_dataset("nyu-mll/glue", "sst2", split="validation")

# 時間短縮のため、最初の200件のみで評価（全体は872件）
# 全件評価する場合は dataset = dataset のままにしてください
eval_dataset = dataset.select(range(200))

# 3. ターゲットトークンのIDを取得
# プロンプトの末尾にスペースを置くため、スペースなしの単語のIDを取得します
pos_token_id = tokenizer.encode("Positive")[0]
neg_token_id = tokenizer.encode("Negative")[0]

print(f"'Positive' のトークンID: {pos_token_id}")
print(f"'Negative' のトークンID: {neg_token_id}\n")

correct_count = 0
total_count = len(eval_dataset)

print("評価を開始します...")

# 4. データセットをループして推論
for item in tqdm(eval_dataset):
    sentence = item["sentence"]
    true_label = item["label"]  # 1: Positive, 0: Negative

    # ゼロショット用のプロンプトを構築
    prompt = f"Review: {sentence}\nSentiment (Positive or Negative): "

    inputs = tokenizer(prompt, return_tensors="pt")

    with torch.no_grad():
        outputs = model(**inputs)

    # 最後のトークンの次に来る予測のロジットを取得
    next_token_logits = outputs.logits[0, -1, :]

    # 'Positive' と 'Negative' のロジットを抽出
    pos_logit = next_token_logits[pos_token_id].item()
    neg_logit = next_token_logits[neg_token_id].item()

    # ロジットが大きい方をモデルの予測とする
    if pos_logit > neg_logit:
        predicted_label = 1 # Positive
    else:
        predicted_label = 0 # Negative

    # 正解判定
    if predicted_label == true_label:
        correct_count += 1

# 5. 正解率の計算と出力
accuracy = correct_count / total_count

print("\n=== 評価結果 ===")
print(f"評価件数: {total_count} 件")
print(f"正解数  : {correct_count} 件")
print(f"正解率  : {accuracy * 100:.2f}%")

SST-2 データセットをロード中...


README.md:   0%|          | 0.00/35.3k [00:00<?, ?B/s]

sst2/train-00000-of-00001.parquet:   0%|          | 0.00/3.11M [00:00<?, ?B/s]

sst2/validation-00000-of-00001.parquet:   0%|          | 0.00/72.8k [00:00<?, ?B/s]

sst2/test-00000-of-00001.parquet:   0%|          | 0.00/148k [00:00<?, ?B/s]

...

Generating validation split:   0%|          | 0/872 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1821 [00:00<?, ? examples/s]

'Positive' のトークンID: 21604
'Negative' のトークンID: 32863

評価を開始します...


100%|██████████| 200/200 [00:52<00:00,  3.78it/s]


=== 評価結果 ===
評価件数: 200 件
正解数  : 107 件
正解率  : 53.50%


## 97. 埋め込みに基づく感情分析

事前学習済み言語モデルでテキストをベクトルで表現（エンコード）し、そのベクトルにフィードフォワード層を通すことで極性ラベルを予測するモデルを学習せよ。

In [6]:
# 1. モデルとトークナイザーの準備
# 軽量で高速なエンコーダ型モデル「DistilBERT」を使用
model_name_97 = "distilbert-base-uncased"
print(f"{model_name_97} をロードしています...\n")

tokenizer_97 = AutoTokenizer.from_pretrained(model_name_97)

# num_labels=2 を指定することで、事前学習済みモデルの上にポジティブ・ネガティブの2クラスに分類するフィードフォワード層（Linear層）が追加される
model_97 = AutoModelForSequenceClassification.from_pretrained(model_name_97, num_labels=2)

# 2. データセットのロードと前処理 (SST-2)
print("データセットをロード・前処理しています...")
dataset = load_dataset("nyu-mll/glue", "sst2")

train_dataset = dataset["train"]
eval_dataset = dataset["validation"]

def tokenize_function(examples):

    return tokenizer_97(
        examples["sentence"],
        padding="max_length",   # 64文字に満たない文は、PADで埋める
        truncation=True,        # 64文字以上の文は、後ろを切り捨てる
        max_length=64
    )

# データセット全体にトークン化を適用
tokenized_train = train_dataset.map(tokenize_function, batched=True)
tokenized_eval = eval_dataset.map(tokenize_function, batched=True)

# 3. 評価指標（正解率: Accuracy）の定義
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    # 出力されたロジットのうち、値が大きい方のインデックスをモデルの最終的な予測値として使用して、正解率を計算
    predictions = np.argmax(logits, axis=-1)
    accuracy = (predictions == labels).mean()
    return {"accuracy": accuracy}

# 4. 学習のハイパーパラメータ設定
training_args = TrainingArguments(
    output_dir="./sentiment_model_results",
    learning_rate=2e-5,                  # ファインチューニングの標準的な学習率
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,                  # 学習データを何周するか
    eval_strategy="epoch",               # エポックごとに評価を実行
    save_strategy="epoch",               # エポックごとにモデルを保存
    logging_steps=10,
    fp16=torch.cuda.is_available(),      # GPUがあれば半精度浮動小数点演算で高速化
)

# 5. Trainerの初期化と学習の実行
trainer = Trainer(
    model=model_97,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_eval,
    compute_metrics=compute_metrics,
)

print("=== 学習を開始します ===\n")
trainer.train()

print("\n=== 学習が完了しました ===")

distilbert-base-uncased をロードしています...



config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

...

Epoch,Training Loss,Validation Loss,Accuracy
1,0.160772,0.287050,0.907110
2,0.119439,0.368495,0.889908
3,0.065013,0.421924,0.902523


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


=== 学習が完了しました ===


## 98. ファインチューニング

問題96のプロンプトに対して、正解の感情ラベルをテキストの応答として返すように事前学習済みモデルをファインチューニングせよ。

In [ ]:
# GPT-2にはパディングトークンがないため、EOSトークンを代用する
tokenizer.pad_token = tokenizer.eos_token

# 3. データ前処理関数の定義
def preprocess_function(examples):
    texts = []
    for sentence, label in zip(examples["sentence"], examples["label"]):
        # ラベルID（0 or 1）を文字列に変換
        label_str = "Positive" if label == 1 else "Negative"

        # 入力プロンプト + 正解ラベル + 終了トークン（EOS）を結合
        text = f"Review: {sentence}\nSentiment (Positive or Negative): {label_str}{tokenizer.eos_token}"
        texts.append(text)

    # テキストをトークン化
    return tokenizer(texts, truncation=True, max_length=128)

print("データをトークン化しています...")
# map関数で一括処理（不要な元のカラムは削除）
tokenized_train = train_dataset.map(preprocess_function, batched=True, remove_columns=train_dataset.column_names)
tokenized_eval = eval_dataset.map(preprocess_function, batched=True, remove_columns=eval_dataset.column_names)

# 4. データコレーターの準備
# mlm=False を指定することで、Causal LM（自己回帰型言語モデル）用のラベルが自動生成されます
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

# 5. 学習用ハイパーパラメータの設定
training_args = TrainingArguments(
    output_dir="./gpt2-sst2-finetuned",
    # overwrite_output_dir=True,
    num_train_epochs=1,               # デモのため1エポック
    per_device_train_batch_size=8,    # バッチサイズ
    per_device_eval_batch_size=8,
    eval_strategy="steps",            # 評価のタイミング
    eval_steps=100,                   # 100ステップごとに評価
    logging_steps=50,
    save_strategy="epoch",
    learning_rate=5e-5,               # 学習率
    report_to="none"                  # wandbなどの外部ログを無効化
)

# 6. Trainerの初期化
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_eval,
    data_collator=data_collator,
)

# 7. ファインチューニングの実行
print("\n=== ファインチューニングを開始します ===")
trainer.train()
print("=== 学習完了 ===\n")

# 8. 学習済みモデルでのテスト推論
print("=== 学習済みモデルでのテスト生成 ===")
model.eval()

test_sentences = [
    "This movie is an absolute masterpiece and I loved every second of it.",
    "Terrible acting, awful plot, completely a waste of time."
]

for sentence in test_sentences:
    prompt = f"Review: {sentence}\nSentiment (Positive or Negative): "
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=2, # "Positive" または "Negative" の生成に十分な長さ
            pad_token_id=tokenizer.eos_token_id,
            do_sample=False   # 貪欲探索（Greedy）で最も確率の高いものを出力
        )

    result = tokenizer.decode(outputs[0], skip_special_tokens=True)
    print("-" * 50)
    print(f"[入力]\n{sentence}")
    print(f"[モデルの出力]\n{result}")

データをトークン化しています...


Map:   0%|          | 0/67349 [00:00<?, ? examples/s]

Map:   0%|          | 0/872 [00:00<?, ? examples/s]

[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.



=== ファインチューニングを開始します ===


Step,Training Loss,Validation Loss
100,2.291649,2.765690
200,2.310959,2.729483
300,2.158639,2.720949
400,2.185783,2.717312
500,2.114191,2.693605
600,2.100159,2.659202
700,2.085549,2.700673
800,2.094022,2.668637
900,2.049255,2.656946
1000,2.032923,2.651897


## 99. 選好チューニング

問題96のプロンプトに対して、正解の感情ラベルを含むテキストを望ましい応答、間違った感情ラベルを含むテキストを望ましくない応答として、事前学習済み言語モデルを選好チューニング (preference tuning) を実施せよ。選好チューニングのアルゴリズムとしては、近傍方策最適化 (PPO: Proximal Policy Optimization) や直接選好最適化 (DPO: Direct Preference Optimization) などが考えられる。


In [ ]:
model_id = "gpt2"
print(f"{model_id} モデルとトークナイザーをロード中...")

# 1. トークナイザーとモデルの準備
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token

# DPOでは学習対象のモデル（Actor）と、比較対象となる元のモデル（Reference）が必要
# DPOTrainerに ref_model=None を渡すと、内部で自動的に元のモデルのコピーを作成してくれます
model = AutoModelForCausalLM.from_pretrained(model_id)


# 3. DPO用のデータセット形式（prompt, chosen, rejected）に変換
def prep_dpo_data(example):
    # プロンプト（入力）
    prompt = f"Review: {example['sentence']}\nSentiment (Positive or Negative):"

    # 正解と不正解の文字列を作成（最後にEOSトークンを付与して生成終了を教える）
    correct_text = f" Positive{tokenizer.eos_token}" if example["label"] == 1 else f"Negative{tokenizer.eos_token}"
    incorrect_text = f" Negative{tokenizer.eos_token}" if example["label"] == 1 else f"Positive{tokenizer.eos_token}"

    return {
        "prompt": prompt,
        "chosen": correct_text,     # 望ましい応答（正解）
        "rejected": incorrect_text  # 望ましくない応答（不正解）
    }

print("DPO用のデータフォーマットに変換中...")
dpo_train = train_dataset.map(prep_dpo_data, remove_columns=train_dataset.column_names)
dpo_eval = eval_dataset.map(prep_dpo_data, remove_columns=eval_dataset.column_names)

# 4. 学習ハイパーパラメータの設定 (TrainingArguments から DPOConfig に変更)
training_args = DPOConfig(
    output_dir="./gpt2-sst2-dpo",
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=1,
    learning_rate=1e-5,
    eval_strategy="steps",
    eval_steps=500,
    logging_steps=500,
    save_strategy="epoch",
    remove_unused_columns=False,
    report_to="none",
    beta=0.1,                    # DPOの温度パラメータ
)

# 5. DPOTrainerの初期化
trainer = DPOTrainer(
    model=model,
    ref_model=None,
    args=training_args,
    train_dataset=dpo_train,
    eval_dataset=dpo_eval,
    processing_class=tokenizer,  # tokenizer から processing_class に名称変更
)


# 6. DPOチューニングの実行
print("\n=== DPO (直接選好最適化) を開始します ===")
trainer.train()
print("=== 学習完了 ===\n")

In [ ]:
# 7. 学習済みモデルでのテスト生成
print("=== DPO学習済みモデルでのテスト生成 ===")
model.eval()

test_sentences = [
    "This movie is an absolute masterpiece and I loved every second of it.",
    "Terrible acting, awful plot, completely a waste of time."
]

for sentence in test_sentences:
    prompt = f"Review: {sentence}\nSentiment (Positive or Negative): "
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=2,
            pad_token_id=tokenizer.eos_token_id,
            do_sample=False
        )

    result = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()
    print("-" * 50)
    print(f"[入力]\n{sentence}")
    print(f"[モデルの出力]\n{result}")